# **Ovarian Reserve - Inference workflow**
___

This notebook runs the **3D instance segmentation inference** workflow in **BiaPy** for ovarian oocytes, following the [ovarian reserve tutorial from the BiaPy documentation](https://biapy.readthedocs.io/en/latest/tutorials/instance_seg/ovarian-reserve.html).

It supports two model-loading modes:
- **Official pretrained checkpoint** recommended for this workflow. It will be automatically downloaded from our [official repository](https://upvehueus-my.sharepoint.com/:u:/g/personal/ignacio_arganda_ehu_eus/IQBqhycRQT1zTakayPKsyJLqAXzibIJSV_xIKADXYnLP3zM?e=0LVuynb&download=1).
- **A BioImage Model Zoo model ID** through BiaPy.

<figure>
<center>
<img src='https://raw.githubusercontent.com/BiaPyX/BiaPy-doc/refs/heads/master/source/img/tutorials/instance-segmentation/ovarian-reserve/F1.large.jpg' width='500'/>
<figcaption>Paper Figure 1: SPIM whole-ovary imaging and model-based oocyte segmentation workflow.</figcaption>
</center>
</figure>

For quick Colab testing, this notebook can reuse the small example dataset from the training workflow and run inference on its validation split.
___

If this workflow is useful for your research, please cite:

*3D Mapping of Intact Ovaries Reveals the Aging Dynamics of the Ovarian Reserve*  
Arturo D'Angelo, Daniel Franco-Barranco, Marco Musy, James Sharpe, Ignacio Arganda-Carreras, Elvan Boke  
bioRxiv 2025.11.07.686728; doi: https://doi.org/10.1101/2025.11.07.686728

## **Expected inputs and outputs**
___

### **Inputs**
This notebook expects:
- **Test Raw Images**: a folder containing the 3D TIFF volumes to segment.
- **[OPTIONAL] Test Label Images**: instance-label volumes matching the raw images, only if you want BiaPy to compute evaluation metrics.
- **Output Folder**: a directory where BiaPy will store predictions, post-processed results and logs.

### **Outputs**
If execution is successful, BiaPy creates result folders with:
- Predicted instance-segmentation channels.
- TIFF instance masks.
- Post-processed instance masks when post-processing is enabled.
- Optional matching statistics if ground truth labels are supplied.

<font color='red'><b>Note</b></font>: for testing purposes, you can also run this notebook with the samples images provided in *Manage file(s) source > Option 3*. Those samples belong to the small training sample set (`oocyte_training.zip`) used in our [tutorial](https://biapy.readthedocs.io/en/latest/tutorials/instance_seg/ovarian-reserve.html).

**Data structure**

To ensure the proper operation of the library, the input data directory tree should be something like this:

```
dataset/
└── test
    ├── raw
    │   ├── test-0001.tif
    │   ├── test-0002.tif
    │   ├── . . .
    │   └── test-9999.tif
    └── label
        ├── test-0001.tif
        ├── test-0002.tif
        ├── . . .
        └── test-9999.tif
```

**⚠️ Warning:** Ensure that images and their corresponding masks are sorted in the same way. A common approach is to fill with zeros the image number added to the filenames (as in the example).

**Input Format Support**

This notebook is compatible with a range of input formats. You can use the following file extensions: `.tif`, `.npy` (every extension for 3D images supported by [scikit-image](https://scikit-image.org/docs/stable/api/skimage.io.html#skimage.io.imread)).

## **Prepare the environment**
___

Establish connection with Google services. You **must be logged in to Google** to continue.
Since this is not Google's own code, Colab may warn about running external code. That is expected for this notebook.

## **Check for GPU access**
---

By default, the session should be using Python 3 and GPU acceleration, but it is possible to ensure that these are set properly by doing the following:

Go to **Runtime -> Change the Runtime type**

**Runtime type: Python 3** *(Python 3 is programming language in which this program is written)*

**Accelerator: GPU** *(Graphics processing unit, choose any of the cards available)*

## **Install BiaPy**
---
This may take a few minutes depending on the current Colab environment.

In [ ]:
#@markdown ##Play to install BiaPy and its dependences
# Install latest release of BiaPy
!pip install biapy==3.6.8

# Then install Pytorch
!pip install torch==2.9.1 torchvision

# Finally install some packages that rely on the Pytorch installation
!pip install timm==1.0.14 pytorch-msssim torchmetrics[image]==1.4.*

import os
import sys
import numpy as np
from tqdm.notebook import tqdm
from skimage.io import imread
import ipywidgets as widgets
from ipywidgets import Output
from biapy import BiaPy

changed_source = False

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.8/50.8 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.4/149.4 kB 6.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.7/594.7 kB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 174.7/174.7 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 66.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 869.5/869.5 kB 59.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 81.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 210.3/210.3 kB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 65.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 88.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.5/26.5 MB 86.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 9.2 MB/s eta 0:00:00
   ━━━━━━

## **Manage file(s) source**
---
The test input folder can be provided using three different options: by directly uploading the folder (option 1), by using a folder stored in your own Google Drive (option 2) or by automatically downloading a few samples of our data (option 3).

Depending on the option chosen, different steps will have to be taken, as explained in the following cells.

### **Option 1: Use your local files and upload them to the notebook**
---
You will be prompted to upload your files to Colab and they will be stored under `/content/input/`.

In [ ]:
#@markdown ##Play the cell to upload local files (test raw images)

from google.colab import files
import os
input_dir = '/content/input/test/x'

if os.path.exists(input_dir):
    # Ask the user if they want to delete the existing items in the folder
    delete_items = ''
    while not delete_items in ['y', 'n']:
        delete_items = input("Do you want to delete the existing items in the folder? (yes[y]/no[n]): ").strip().lower()
    if delete_items == 'y':
        for delete_root, delete_dirs, delete_files in os.walk(input_dir, topdown=False):
            for name in delete_files:
                os.unlink(os.path.join(delete_root, name))
else:
    # Ensure the directory exists
    os.makedirs(input_dir, exist_ok=True)


%cd {input_dir}
uploaded = files.upload()
%cd /content

In [ ]:
#@markdown ## [OPTIONAL] Play the cell to upload local files (test labels)

from google.colab import files
import os
input_dir = '/content/input/test/y'

if os.path.exists(input_dir):
    # Ask the user if they want to delete the existing items in the folder
    delete_items = ''
    while not delete_items in ['y', 'n']:
        delete_items = input("Do you want to delete the existing items in the folder? (yes[y]/no[n]): ").strip().lower()
    if delete_items == 'y':
        for delete_root, delete_dirs, delete_files in os.walk(input_dir, topdown=False):
            for name in delete_files:
                os.unlink(os.path.join(delete_root, name))
else:
    # Ensure the directory exists
    os.makedirs(input_dir, exist_ok=True)


%cd {input_dir}
uploaded = files.upload()
%cd /content

### **Option 2: Mount your Google Drive**
---
To use this notebook on your own data from Google Drive, you need to mount Google Drive first.

Play the cell below to mount your Google Drive and follow the link that will be shown. In the new browser window, select your drive and select 'Allow', copy the code, paste into the cell and press enter. This will give Colab access to the data on the drive.

Once this is done, your data will be available in the **Files** tab on the top left of notebook.

In [ ]:
#@markdown ##Play the cell to connect your Google Drive to Colab

#@markdown * Click on the URL.

#@markdown * Sign in your Google Account.

#@markdown * Copy the authorization code.

#@markdown * Enter the authorization code.

#@markdown * Click on "Files" site on the right. Refresh the site. Your Google Drive folder should now be available here as "drive".

# mount user's Google Drive to Google Colab.
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


### **Option 3: Download our publication's test samples**
---
Don't you have data readily available but still want to test the notebook? No problem! Simply execute the following cell to download a sample dataset.

Specifically, we'll use the **validation set used to evaluate the model** in our manuscript (`oocyte_training.zip`), as described in our [tutorial](https://biapy.readthedocs.io/en/latest/tutorials/instance_seg/ovarian-reserve.html).

In [ ]:
#@markdown ## Play to download the example ovarian reserve dataset
import os
os.chdir('/content')
dataset_zip = 'oocyte_training.zip'
dataset_dir = '/content/oocyte_training'

print('Downloading example ovarian reserve dataset...')
if not os.path.exists(dataset_zip) and not os.path.exists(dataset_dir):
    !wget -q -O oocyte_training.zip 'https://upvehueus-my.sharepoint.com/:u:/g/personal/ignacio_arganda_ehu_eus/IQBlTg1-y8MlSqwgDpLZuPAgAU5oE0HOqc6vjDK7vVh_xBM?e=MMgzZf&download=1'
if os.path.exists(dataset_zip) and not os.path.exists(dataset_dir):
    !unzip -q oocyte_training.zip
print('Example dataset ready at: /content/oocyte_training')
print('Use /content/oocyte_training/val/raw as inference input and /content/oocyte_training/val/label as optional labels.')

Example dataset ready at: /content/oocyte_training
Use /content/oocyte_training/val/raw as inference input and /content/oocyte_training/val/label as optional labels.


## **Paths for Input Images and Output Files**
___

Depending on the option you chose for managing file sources, you'll set your paths differently:

- **Option 1 (Upload from Local Machine)**:
  - Set `test_data_path` to `/content/input/test/raw`
  - Set `test_data_gt_path` to `/content/input/test/label`
  
- **Option 2 (Use Google Drive Data)**:
  - Insert the paths to your input files and your desired output directory here, i.e., `/content/gdrive/MyDrive/...`.
- **Option 3 (Use Our Sample Data)**:
  - Set `train_data_path` to `/content/oocyte_training/val/raw`
  - Set `train_data_gt_path` to `/content/oocyte_training/val/label`
  - Set `output_path` to `/content/out`

**Note**: Ensure you download your results from the `/content/out` directory after the process!

**Helpful Tip**: If you're unsure about the paths to your folders, look at the top left of this notebook for a small folder icon. Navigate through the directories until you locate your desired folder. Right-click on it and select "Copy Path" to copy the folder's path.

In [ ]:
#@markdown ##### Path to test raw images
test_data_path = '/content/oocyte_training/val/raw' #@param {type:"string"}
#@markdown ##### Select to load ground truth labels and compute metrics
load_gt = True #@param {type:"boolean"}
#@markdown ##### Path to optional test labels
test_data_gt_path = '/content/oocyte_training/val/label' #@param {type:"string"}
#@markdown ##### Output folder
output_path = '/content/output' #@param {type:"string"}

import os
from pathlib import Path

def count_image_files(directory):
    if not directory or not os.path.exists(directory):
        return 0
    image_extensions = {'.jpg', '.jpeg', '.png', '.tif', '.tiff', '.npy', '.h5', '.hd5', '.zarr'}
    count = 0
    for root, dirs, files in os.walk(directory):
        for file in files:
            if Path(file).suffix.lower() in image_extensions:
                count += 1
    return count

num_test_images = count_image_files(test_data_path)
num_test_labels = count_image_files(test_data_gt_path) if load_gt else 0
print(f'Number of test raw images: {num_test_images}')
if load_gt:
    print(f'Number of test label images: {num_test_labels}')
    if num_test_images != num_test_labels:
        print('Warning: the number of raw images and labels does not match.')

Number of test raw images: 7
Number of test label images: 7


## **Dataset Visualization**
---

In [ ]:
#@markdown ## Play to visualize the test data
# @markdown Use the *Image index* and *Z value* scrolls to navigate among volumes and slices. **Note**: it might take a few seconds to refresh the images.
%matplotlib inline
import numpy as np
from matplotlib import pyplot as plt
from skimage.io import imread
import os
from ipywidgets import IntSlider, Layout, VBox, Output

ids_input_all = sorted(next(os.walk(test_data_path))[2]) if os.path.exists(test_data_path) else []
ids_input = [f for f in ids_input_all if f.lower().endswith(('.tif', '.tiff'))]

if len(ids_input) == 0:
    print('No TIFF images found in the selected test path.')
else:
    input_img = imread(os.path.join(test_data_path, ids_input[0]))
    ids_gt = []
    gt_img = None
    if load_gt and os.path.exists(test_data_gt_path):
        ids_gt_all = sorted(next(os.walk(test_data_gt_path))[2])
        ids_gt = [f for f in ids_gt_all if f.lower().endswith(('.tif', '.tiff'))]
        if len(ids_gt) > 0:
            gt_img = imread(os.path.join(test_data_gt_path, ids_gt[0])).astype(np.uint16)

    vals = np.linspace(0, 1, 256)
    np.random.shuffle(vals)
    cmap = plt.cm.colors.ListedColormap(plt.cm.gist_rainbow(vals))
    cmap.colors[0] = [0.0, 0.0, 0.0, 1.0]

    sample_slider = IntSlider(value=1, min=1, max=len(ids_input), step=1, description='Image index:', continuous_update=False, layout=Layout(width='500px'))
    z_slider = IntSlider(value=1, min=1, max=len(input_img), step=1, description='Z value:', continuous_update=False, layout=Layout(width='500px'))
    sample_slider.style.description_width = 'initial'
    z_slider.style.description_width = 'initial'
    output = Output()
    state = {'input_img': input_img, 'gt_img': gt_img, 'index': 0}

    def update_sample(change):
        idx = change['new'] - 1
        state['index'] = idx
        state['input_img'] = imread(os.path.join(test_data_path, ids_input[idx]))
        if load_gt and len(ids_gt) > idx:
            state['gt_img'] = imread(os.path.join(test_data_gt_path, ids_gt[idx])).astype(np.uint16)
        else:
            state['gt_img'] = None
        z_slider.max = len(state['input_img'])
        z_slider.value = 1
        display_slice({'new': 1})

    def display_slice(change):
        with output:
            output.clear_output(wait=True)
            z = change['new'] - 1
            plt.figure(figsize=(12, 6))
            plt.subplot(1, 2, 1)
            plt.title(f'Raw image: {ids_input[state["index"]]} | z={z+1}')
            plt.imshow(state['input_img'][z], cmap='gray')
            plt.axis('off')
            if state['gt_img'] is not None:
                plt.subplot(1, 2, 2)
                plt.title('Label')
                plt.imshow(state['gt_img'][z], cmap=cmap, interpolation='nearest')
                plt.axis('off')
            plt.show()

    controls = VBox([sample_slider, z_slider])
    display(controls, output)
    sample_slider.observe(update_sample, names='value')
    z_slider.observe(display_slice, names='value')
    display_slice({'new': 1})

Output()

## **Select pre-trained model to use**
---

In [ ]:
# @markdown ###OPTIONAL: Check BioImage Model Zoo (BMZ) models compatible with BiaPy
# @markdown Use this option to generate a full list of the available BiaPy-compatible models in the BMZ.

# @markdown **Important:** To select one of the listed models (if any), you will have to run the next cell and select "BioImage Model Zoo" as the source of the model. Then, paste the corresponding model's nickname into the created field.
# @markdown <div><img src="https://bioimage.io/static/img/bioimage-io-logo.svg" width="600"/></div>


import json
import pooch
from IPython.display import HTML, display
from tqdm.notebook import tqdm

import requests
from biapy.models import check_bmz_model_compatibility, find_bmz_models

# Change pooch verbosity
logger = pooch.get_logger()
logger.setLevel("WARNING")

# Check the models that BiaPy can consume
models = find_bmz_models()

# Check axes, preprocessing functions used and postprocessing.
pytorch_models = []
imposed_vars = []

workflow_specs = {
    "workflow_type": "INSTANCE_SEG",
    "ndim": "3D",
    "nclasses": "all",
}
if len(models) > 0:
    print("Analizing models from the Zoo . . .")

for model in tqdm(models, total=len(models)):
    try:
        (
            preproc_info,
            error,
            error_message,
            model_imposed_vars
        ) = check_bmz_model_compatibility(model, workflow_specs=workflow_specs)
    except:
        error = True

    if not error:
        imposed_vars.append(model_imposed_vars)
        pytorch_models.append(model)

# Print the possible models
html = "<table style='width:100%''>"
c = 0
for i, model in enumerate(pytorch_models):

    nickname = "Nick not found"
    if 'nickname' in model:
        nickname = model['nickname']
    elif 'alias' in model:
        nickname = model['alias']

    nickname_icon = "Emoji not found"
    if 'id_emoji' in model["raw"]["manifest"]:
        nickname_icon = model["raw"]["manifest"]['id_emoji']

    cover_url = "https://hypha.aicell.io/bioimage-io/artifacts/{}/files/{}".format(nickname, model["raw"]["manifest"]['covers'][0])
    restrictions = ""
    for key, val in imposed_vars[i].items():
        if key == 'MODEL.N_CLASSES':
            restrictions += "<p>number_of_classes: {}</p>".format(val)
        elif key == "PROBLEM.INSTANCE_SEG.DATA_CHANNELS":
            problem_channels = 'Binary mask + Contours'
            if val == "BC":
                problem_channels = "Binary mask + Contours"
            elif val == 'BP':
                problem_channels = "Binary mask + Central points"
            elif val == 'BD':
                problem_channels = "Binary mask + Distance map"
            elif val == 'BCM':
                problem_channels = "Binary mask + Contours + Foreground mask"
            elif val == 'BCD':
                problem_channels = "Binary mask + Contours + Distance map"
            elif val == 'BCDv2':
                problem_channels = "Binary mask + Contours + Distance map with background"
            elif val == 'Dv2':
                problem_channels = "Distance map with background"
            restrictions += "<p>Problem representation: {}</p>".format(problem_channels)
    if c == 0:
        html += "<tr>"
    html += "<td style='width:33%'>"
    html += "<p style='color:#2196f3'>%s</p><p>Nickname: %s (%s)</p>%s<img src='%s' height='130'></td>"%(
        model["raw"]["manifest"]['name'],
        nickname,
        nickname_icon,
        restrictions,
        cover_url,
    )
    c +=1
    if c == 3:
        html += "</tr>"
        c=0

html += "</table>"
if len( pytorch_models ) == 0:
    display(HTML('<h1>No BMZ models compatible with BiaPy were found for this task.</h1><br>'))
else:
    display(HTML('<h1>List of models that can be used in BiaPy:</h1><br>'))
    display(HTML(html))


Analizing models from the Zoo . . .


  0%|          | 0/135 [00:00<?, ?it/s]

Neuron Segmentation in EM (Membrane Prediction)Nickname: impartial-shrimp (🦐)Problem representation: Binary mask + Contours,PlatynereisEMnucleiSegmentationBoundaryModelNickname: organized-badger (🦡)Problem representation: Binary mask + Contours,MitochondriaEMSegmentationBoundaryModelNickname: kind-seashell (🐚)Problem representation: Binary mask + Contours
PlatynereisEMcellsSegmentationBoundaryModelNickname: willing-hedgehog (🦔)Problem representation: Binary mask + Contours,EnhancerMitochondriaEM3DNickname: independent-shrimp (🦐)Problem representation: Binary mask + Contours,EM3DBoundaryEnhancerNickname: determined-chipmunk (🐿)Problem representation: Binary mask + Contours
3D Attention U-Net for mitochondria segmentation in EMNickname: good-butterfly (🦋)Problem representation: Binary mask + Contours,3D Attention U-Net for neuron segmentation in EMNickname: zealous-snail (🐌)Problem representation: Binary mask + Contours,3D ResUNetpp for neuron segmentation in EMNickname: exuberant-parrot (🦜)Problem representation: Binary mask + Contours
3D Residual U-Net for mitochondria segmentation in EMNickname: conscientious-dromedary (🐪)Problem representation: Binary mask + Contours,3D Residual U-Net for cyst segmentation in fluorescenceNickname: merry-water-buffalo (🐃)Problem representation: Binary mask + Contours,3D Residual Squeeze-and-Excitation U-Net for cyst segmentation in fluorescenceNickname: venomous-swan (🦢)Problem representation: Binary mask + Contours
3D Residual U-Net for neuron segmentation in EMNickname: willing-whale (🐳)Problem representation: Binary mask + Contours,3D U-Net for cyst segmentation in fluorescenceNickname: happy-honeybee (🐝)Problem representation: Binary mask + Contours,3D ResUNetpp for cyst segmentation in fluorescenceNickname: intelligent-lion (🦁)Problem representation: Binary mask + Contours
3D Squeeze-and-Excitation U-Net for cyst segmentation in fluorescenceNickname: idealistic-turtle (🐢)Problem representation: Binary mask + Contours,3D Attention U-Net for cyst segmentation in fluorescenceNickname: heroic-otter (🦦)Problem representation: Binary mask + Contours,3D Residual U-Net for neurite segmentation in EMNickname: loving-monkey (🐒)Problem representation: Binary mask + Contours


In [ ]:
#@markdown ###Play to select the source to build the model (BiaPy or BioImage Model Zoo) { run: "auto", vertical-output: true, display-mode: "form" }

#@markdown **BiaPy**: to use the model implemented in BiaPy.

#@markdown **Bioimage Model Zoo (BMZ)**: to use models from the [BMZ repository](https://bioimage.io/#/). You can run the above cell to generate an updated list of the models that can be used with BiaPy. Copy the nickname from the model and paste it below.
import ipywidgets as widgets
from ipywidgets import Output

changed_source = True
exists_bmz = False
# create widgets
source = widgets.ToggleButtons(
    options=['BiaPy', 'BioImage Model Zoo'],
    description='Source:',
    disabled=False,
    button_style='', # 'success', 'info', 'warning', 'danger' or ''
    tooltips=['Models created during this workflow', 'BioImage Model Zoo model'],
#     icons=['check'] * 3
)

bmz = widgets.Text(
    # value='10.5281/zenodo.5764892',
    placeholder='Nickname of BMZ model',
    description='ID:',
    disabled=False
)

# display the first widget
display(source)

# intialize the output - second widget
out = Output()

def changed(change):
    '''
    Monitor change in the first widget
    '''
    global out
    global exists_bmz
    if source.value == 'BiaPy':
        bmz.layout.display = 'none'
        out.clear_output() #clear output
        out = Output() # redefine output
    else:
        bmz.layout.display = 'none'
        bmz.layout.display = 'flex'
        if not exists_bmz:
          out.append_display_data(bmz)
          display(out)
        exists_bmz = True

# monitor the source widget for changes
source.observe(changed, 'value')

ToggleButtons(description='Source:', options=('BiaPy', 'BioImage Model Zoo'), tooltips=('Models created during…

## **Download pretrained model and apply 3D segmentation workflow**


In [23]:
#@markdown ##Play to download and run the selected pretrained model
import os
import errno
import yaml
import gdown

os.chdir('/content/')

checkpoint_file = "/content/ovarian_reserve_pretrained.pth"
checkpoint_url = "https://upvehueus-my.sharepoint.com/:u:/g/personal/ignacio_arganda_ehu_eus/IQBqhycRQT1zTakayPKsyJLqAXzibIJSV_xIKADXYnLP3zM?e=0LVuyn"

job_name = 'ovarian_reserve_inference'
template_file = '/content/ovarian_reserve_inference_template.yaml'
inference_file = f'/content/{job_name}.yaml'
template_url = 'https://raw.githubusercontent.com/BiaPyX/BiaPy/master/templates/instance_segmentation/Ovarian_Reserve_paper/ovarian_reserve_inference.yaml'


# remove template file it is exists
template_file = '{}.yaml'.format(job_name)
if os.path.exists( template_file ):
    os.remove( template_file )

# Download .yaml file and model weights
if not os.path.exists( inference_file ):
    !wget -q -O {template_file} {template_url}
    print("\Ovarian reserve yaml configuration file downloaded!")
if changed_source:
    if not os.path.exists( checkpoint_file ) and source.value == "BiaPy":
        !wget -q -O {checkpoint_file} {checkpoint_url}+"&download=1"
        print( '\n Pretrained model weights successfully downloaded!')


# Check folders before modifying the .yaml file
if not os.path.exists(test_data_path):
    raise FileNotFoundError(errno.ENOENT, os.strerror(errno.ENOENT), test_data_path)
ids = sorted(next(os.walk(test_data_path))[2])
if len(ids) == 0:
    raise ValueError("No images found in dir {}".format(test_data_path))

if not os.path.exists(template_file):
    raise ValueError("No YAML configuration file found in {}".format(template_file))

if source.value == "BiaPy" and not os.path.exists(checkpoint_file):
    raise ValueError("No checkpoint file found in {}".format(checkpoint_file))


# open template configuration file
with open( template_file, 'r') as stream:
    try:
        biapy_config = yaml.safe_load(stream)
    except yaml.YAMLError as exc:
        print(exc)

# change source to build model - biapy or bmz
if changed_source:
    if source.value == "BiaPy":
        biapy_config['MODEL']['SOURCE'] = "biapy"
    elif source.value == 'BioImage Model Zoo':
        checkpoint_file = ''
        biapy_config['PATHS']['CHECKPOINT_FILE'] = ''
        biapy_config['MODEL']['LOAD_CHECKPOINT'] = False
        biapy_config['MODEL']['SOURCE'] = "bmz"
        biapy_config['MODEL']['BMZ'] = {}
        biapy_config['MODEL']['BMZ']['SOURCE_MODEL_ID'] = str(bmz.value).strip()
else:
    biapy_config['MODEL']['SOURCE'] = "biapy"


biapy_config['DATA']['TEST']['PATH'] = test_data_path
biapy_config['DATA']['TEST']['GT_PATH'] = test_data_gt_path

biapy_config['DATA']['TEST']['LOAD_GT'] = load_gt
biapy_config['TRAIN']['ENABLE'] = False
# model weights
if checkpoint_file != '':
    biapy_config['PATHS'] = {}
    biapy_config['PATHS']['CHECKPOINT_FILE'] = checkpoint_file
    biapy_config['MODEL'] = {}
    biapy_config['MODEL']['LOAD_CHECKPOINT'] = True

# do not use chunks in Colab
biapy_config['TEST']['BY_CHUNKS']['ENABLE'] = False

# save file
with open( template_file, 'w') as outfile:
    yaml.dump(biapy_config, outfile, default_flow_style=False)

print( "Inference configuration finished.")

job_name = os.path.splitext(template_file)[0].split('/')[-1]

# Run the code
biapy = BiaPy(template_file, result_dir=output_path, name=job_name, run_id=1, gpu=0)
biapy.run_job()



<>:26: SyntaxWarning: invalid escape sequence '\O'
<>:26: SyntaxWarning: invalid escape sequence '\O'
/tmp/ipykernel_1640/2197209501.py:26: SyntaxWarning: invalid escape sequence '\O'
  print("\Ovarian reserve yaml configuration file downloaded!")


[19:17:42.219636] \Ovarian reserve yaml configuration file downloaded!
[19:17:49.040709] 
 Pretrained model weights successfully downloaded!
[19:17:49.053514] Inference configuration finished.
[19:17:49.053911] Date     : 2026-04-01 19:17:49
[19:17:49.054028] Arguments: Namespace(config='ovarian_reserve_inference.yaml', result_dir='/content/output', name='ovarian_reserve_inference', run_id=1, gpu=0, world_size=1, local_rank=-1, dist_on_itp=False, dist_url='env://', dist_backend='nccl')
[19:17:49.054089] Job      : ovarian_reserve_inference_1
[19:17:49.054131] BiaPy    : 3.6.8
[19:17:49.054172] Python   : 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
[19:17:49.054227] PyTorch  : 2.9.1+cu128
[19:17:49.062058] The following changes were made in order to adapt the input configuration:
[19:17:49.066671] Not using distributed mode
[19:17:49.070170] Configuration details:
[19:17:49.070252] AUGMENTOR:
  AFFINE_MODE: reflect
  AUG_NUM_SAMPLES: 10
  AUG_SAMPLES: True
  BRIGHTNESS: False
  B

[19:19:20.249492] ### MERGE-3D-OV-CROP ###
[19:19:20.249571] Merging (200, 40, 128, 128, 3) images into (40, 256, 256, 3) with overlapping . . .
[19:19:20.249617] Minimum overlap selected: (0, 0, 0)
[19:19:20.249658] Padding: (10, 50, 50)
[19:19:20.255463] Real overlapping (%): (0.0, 0.07142857142857142, 0.07142857142857142)
[19:19:20.255557] Real overlapping (pixels): (0.0, 2.0, 2.0)
[19:19:20.255597] (2, 10, 10) patches per (z,y,x) axis
[19:19:20.450054] **** New data shape is: (40, 256, 256, 3)
[19:19:20.450181] ### END MERGE-3D-OV-CROP ###
[19:19:20.485383] ### MERGE-3D-OV-CROP ###
[19:19:20.485501] Merging (200, 40, 128, 128, 3) images into (40, 256, 256, 3) with overlapping . . .
[19:19:20.485558] Minimum overlap selected: (0, 0, 0)
[19:19:20.485601] Padding: (10, 50, 50)
[19:19:20.491599] Real overlapping (%): (0.0, 0.07142857142857142, 0.07142857142857142)
[19:19:20.491749] Real overlapping (pixels): (0.0, 2.0, 2.0)
[19:19:20.491797] (2, 10, 10) patches per (z,y,x) axis
[19:19:

[19:19:21.227465] Creating instances with watershed . . .
[19:19:22.403599] Thresholds used: {'seed': [0.470703125, 0.342529296875, 0.4111328125], 'growth_mask': [0.2353515625]}
[19:19:22.497900] Saving (1, 40, 256, 256, 1) data as .tif in folder: /content/output/ovarian_reserve_inference/results/ovarian_reserve_inference_1/per_image_instances


[19:19:22.507939] Calculating matching stats . . .
[19:19:22.508048] Its respective image seems to be: /content/oocyte_training/val/label/5W_130858_frame89.tif


...

[19:28:56.089352] Calculating matching stats . . .
[19:28:56.089482] Its respective image seems to be: /content/oocyte_training/val/label/40W_094116_1.tif


[19:28:57.921200] DatasetMatching: {'criterion': 'iou', 'thresh': 0.3, 'fp': 0, 'tp': 8, 'fn': 0, 'precision': 1.0, 'recall': 1.0, 'accuracy': 1.0, 'f1': 1.0, 'n_true': 8, 'n_pred': 8, 'mean_true_score': np.float32(0.8761581), 'mean_matched_score': np.float32(0.8761581), 'panoptic_quality': np.float32(0.8761581)}
[19:28:57.923673] DatasetMatching: {'criterion': 'iou', 'thresh': 0.5, 'fp': 0, 'tp': 8, 'fn': 0, 'precision': 1.0, 'recall': 1.0, 'accuracy': 1.0, 'f1': 1.0, 'n_true': 8, 'n_pred': 8, 'mean_true_score': np.float32(0.8761581), 'mean_matched_score': np.float32(0.8761581), 'panoptic_quality': np.float32(0.8761581)}
[19:28:57.925513] DatasetMatching: {'criterion': 'iou', 'thresh': 0.75, 'fp': 1, 'tp': 7, 'fn': 1, 'precision': 0.875, 'recall': 0.875, 'accuracy': 0.7777777777777778, 'f1': 0.875, 'n_true': 8, 'n_pred': 8, 'mean_true_score': np.float32(0.78277063), 'mean_matched_score': np.float32(0.894595), 'panoptic_quality': np.float32(0.78277063)}
[19:28:57.925655] Checking the p

[19:28:58.089265] Removed 0 instances by properties ([['npixels'], ['sphericity', 'npixels']]), 8 instances left
[19:28:58.094343] Saving (1, 40, 256, 256, 1) data as .tif in folder: /content/output/ovarian_reserve_inference/results/ovarian_reserve_inference_1/per_image_post_processing


[19:28:58.107144] Calculating matching stats after post-processing . . .


[19:28:59.953367] DatasetMatching: {'criterion': 'iou', 'thresh': 0.3, 'fp': 0, 'tp': 8, 'fn': 0, 'precision': 1.0, 'recall': 1.0, 'accuracy': 1.0, 'f1': 1.0, 'n_true': 8, 'n_pred': 8, 'mean_true_score': np.float32(0.8761581), 'mean_matched_score': np.float32(0.8761581), 'panoptic_quality': np.float32(0.8761581)}
[19:28:59.956220] DatasetMatching: {'criterion': 'iou', 'thresh': 0.5, 'fp': 0, 'tp': 8, 'fn': 0, 'precision': 1.0, 'recall': 1.0, 'accuracy': 1.0, 'f1': 1.0, 'n_true': 8, 'n_pred': 8, 'mean_true_score': np.float32(0.8761581), 'mean_matched_score': np.float32(0.8761581), 'panoptic_quality': np.float32(0.8761581)}
[19:28:59.958150] DatasetMatching: {'criterion': 'iou', 'thresh': 0.75, 'fp': 1, 'tp': 7, 'fn': 1, 'precision': 0.875, 'recall': 0.875, 'accuracy': 0.7777777777777778, 'f1': 0.875, 'n_true': 8, 'n_pred': 8, 'mean_true_score': np.float32(0.78277063), 'mean_matched_score': np.float32(0.894595), 'panoptic_quality': np.float32(0.78277063)}
[19:28:59.958358] Releasing memo

## **Visualize instance segmentation results**
---

In [ ]:
#@markdown ###Play to visualize results from the test set
#@markdown The current model will be applied to some test images and results will be shown as browsable 2D stacks displaying:
#@markdown 1. The **Source image**.
#@markdown 2. Its corresponding **Ground truth** labels.
#@markdown 3. The model **Prediction** labels using watershed.
#@markdown 4. The model **Post-processed** labels (using size and shape filters).
#@markdown 5. The **Overlay** between ground truth and post-processed labels.

%matplotlib inline
import matplotlib
import numpy as np
from numpy.random import randint, seed
from matplotlib import pyplot as plt
from ipywidgets import interact, fixed
import ipywidgets as widgets
from google.colab import output
output.enable_custom_widget_manager()

final_results = os.path.join(output_path, job_name, 'results', job_name + "_1")

instance_results = os.path.join(final_results, "per_image_instances")
post_results = os.path.join(final_results, "per_image_post_processing")

show_post = os.path.exists( post_results )

# Show a few examples to check that they have been stored correctly
ids_pred_all = sorted(next(os.walk(instance_results))[2])
ids_pred = [f for f in ids_pred_all if f.lower().endswith(('.tif', '.tiff'))] # Filter only image files
ids_input = sorted(next(os.walk(test_data_path))[2])
ids_gt = sorted(next(os.walk(test_data_gt_path))[2])

# create random color map
vals = np.linspace(0,1,256)
np.random.shuffle(vals)
cmap = plt.cm.colors.ListedColormap(plt.cm.gist_rainbow(vals))
cmap.colors[0] = [0., 0., 0., 1.] # set background to black

samples_to_show = min(len(ids_input), 3)
chosen_images = np.random.choice(len(ids_input), samples_to_show, replace=False)
seed(1)

test_samples = []
test_sample_preds = []
test_sample_posts = []
test_sample_gt = []

# read 3D images again
for i in range(len(chosen_images)):
    aux = imread(os.path.join(test_data_path, ids_input[chosen_images[i]]))
    test_samples.append(aux)

    # Use the filename from ids_input to find the corresponding predicted file
    pred_filename = ids_input[chosen_images[i]]
    if pred_filename in ids_pred: # Ensure the predicted file actually exists
        aux = imread(os.path.join(instance_results, pred_filename)).astype(np.uint16)
        test_sample_preds.append(aux)
    else:
        print(f"Warning: Predicted file {pred_filename} not found in instance_results. Skipping.")
        continue # Skip this iteration if predicted file is missing

    if show_post:
        if pred_filename in ids_pred: # Check again for post-processed file
            aux = imread(os.path.join(post_results, pred_filename)).astype(np.uint16)
            test_sample_posts.append(aux)
        else:
            print(f"Warning: Post-processed file {pred_filename} not found in post_results. Skipping.")
            test_sample_posts.append(None) # Append None or handle as appropriate

    aux = imread(os.path.join(test_data_gt_path, ids_gt[chosen_images[i]])).astype(np.uint16)
    test_sample_gt.append(aux)

# Adjust samples_to_show based on what was actually loaded successfully
# This might be tricky if some predicted/post-processed files are missing
# For simplicity, we'll assume a direct match for now and let the loop handle skips.

num_subplots = 5 if show_post else 4

# function to show results in 3D within a widget
def scroll_in_z(z, j):

    plt.figure(figsize=(25,5))
    # Source
    plt.subplot(1,num_subplots,1)
    plt.axis('off')
    plt.imshow(test_samples[j][z-1], cmap='gray')
    plt.title('Source (z = ' + str(z) + ')', fontsize=15)

    # Target (Ground-truth)
    plt.subplot(1,num_subplots,2)
    plt.axis('off')
    plt.imshow(test_sample_gt[j][z-1], cmap=cmap, interpolation='nearest')
    plt.title('Ground truth (z = ' + str(z) + ')', fontsize=15)

    # Prediction
    plt.subplot(1,num_subplots,3)
    plt.axis('off')
    plt.imshow(test_sample_preds[j][z-1], cmap=cmap, interpolation='nearest')
    plt.title('Prediction (z = ' + str(z) + ')', fontsize=15)

    # Post-processed
    if show_post and test_sample_posts[j] is not None:
        plt.subplot(1,num_subplots,4)
        plt.axis('off')
        plt.imshow(test_sample_posts[j][z-1], cmap=cmap, interpolation='nearest')
        plt.title('Post-processed (z = ' + str(z) + ')', fontsize=15)

    # Overlay
    plt.subplot(1,num_subplots,num_subplots)
    plt.axis('off')
    plt.imshow(test_sample_gt[j][z-1], cmap='Greens')
    if show_post and test_sample_posts[j] is not None:
        plt.imshow(test_sample_posts[j][z-1], alpha=0.5, cmap='Purples')
    else:
        plt.imshow(test_sample_preds[j][z-1], alpha=0.5, cmap='Purples') # Fallback to prediction if post-processed not available
    plt.title('Overlay (z = ' + str(z) + ')', fontsize=15)

    plt.show()

# Only iterate up to the number of successfully loaded samples
for j in range(len(test_samples)):
    interact(scroll_in_z, z=widgets.IntSlider(min=1, max=test_sample_gt[j].shape[0], step=1, value=test_sample_gt[j].shape[0]//2), j=fixed(j));

interactive(children=(IntSlider(value=20, description='z', max=40, min=1), Output()), _dom_classes=('widget-in…

interactive(children=(IntSlider(value=20, description='z', max=40, min=1), Output()), _dom_classes=('widget-in…

interactive(children=(IntSlider(value=20, description='z', max=40, min=1), Output()), _dom_classes=('widget-in…

## **Download instance segmentation results**
---

In [14]:
#@markdown ###Play to download a zip file with all instance segmentation results (after post-processing).

from google.colab import files

!zip -q -j /content/ovarian_reserve_results.zip $post_results/*.tif

files.download("/content/ovarian_reserve_results.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## **Acknowledgments**
---
We would like to express our gratitude for the inspiration drawn from the exceptional [ZeroCostDL4Mic notebooks](https://github.com/HenriquesLab/ZeroCostDL4Mic/wiki). Specifically, we've adopted some of their metric and parameter descriptions, as well as the 3D visualization widget code present in their [U-Net 3D notebook](https://colab.research.google.com/github/HenriquesLab/ZeroCostDL4Mic/blob/master/Colab_notebooks/U-Net_3D_ZeroCostDL4Mic.ipynb). Additionally, our heartfelt thanks go out to [Estibaliz Gomez-de-Mariscal](https://scholar.google.es/citations?user=gsg-TAUAAAAJ) for her invaluable support and perceptive feedback, which significantly enhanced the quality of this notebook.
